In [10]:
"""
BRONZE LAYER
------------
Land the raw CSV into DuckDB exactly as it arrived. No cleaning.
"""

import duckdb
import pandas as pd
from datetime import datetime

RAW_CSV_PATH = "Sample - Superstore.csv"   # TODO: update to your actual path
DB_PATH = "warehouse.duckdb"

def run():
    # TODO: read the CSV with pandas.
    # Hint: this file has non-UTF8 characters, so you'll likely need
    # encoding="latin1" or you'll get a UnicodeDecodeError.
    df = pd.read_csv(RAW_CSV_PATH, encoding="latin1")  # <-- fix this line

    # TODO: add two metadata columns to df:
    #   _loaded_at   -> current timestamp (datetime.now())
    #   _source_file -> just the filename, e.g. "Sample_-_Superstore.csv"
    df["_loaded_at"] = datetime.now()
    df["_source_file"] = "Sample - Superstore.csv"

    con = duckdb.connect(DB_PATH)

    # TODO: write df into a DuckDB table called bronze_sales.
    # Hint: CREATE OR REPLACE TABLE bronze_sales AS SELECT * FROM df
    # (CREATE OR REPLACE makes this idempotent - safe to re-run anytime)
    con.execute("CREATE OR REPLACE TABLE bronze_sales AS SELECT * FROM df")

    row_count = con.execute("SELECT COUNT(*) FROM bronze_sales").fetchone()[0]
    print(f"Bronze load complete: {row_count} rows landed in bronze_sales")

    con.close()

if __name__ == "__main__":
    run()

Bronze load complete: 9994 rows landed in bronze_sales


In [11]:
import duckdb
con = duckdb.connect("warehouse.duckdb")
print(con.execute("DESCRIBE bronze_sales").fetchdf())
print(con.execute('SELECT "Order Date", "Ship Date", _loaded_at FROM bronze_sales LIMIT 3').fetchdf())
con.close()

      column_name   column_type null   key default extra
0          Row ID        BIGINT  YES  None    None  None
1        Order ID       VARCHAR  YES  None    None  None
2      Order Date       VARCHAR  YES  None    None  None
3       Ship Date       VARCHAR  YES  None    None  None
4       Ship Mode       VARCHAR  YES  None    None  None
5     Customer ID       VARCHAR  YES  None    None  None
6   Customer Name       VARCHAR  YES  None    None  None
7         Segment       VARCHAR  YES  None    None  None
8         Country       VARCHAR  YES  None    None  None
9            City       VARCHAR  YES  None    None  None
10          State       VARCHAR  YES  None    None  None
11    Postal Code        BIGINT  YES  None    None  None
12         Region       VARCHAR  YES  None    None  None
13     Product ID       VARCHAR  YES  None    None  None
14       Category       VARCHAR  YES  None    None  None
15   Sub-Category       VARCHAR  YES  None    None  None
16   Product Name       VARCHAR